<a href="https://colab.research.google.com/github/PilliSrilekha1419/ai-mentor-portfolio/blob/main/Day10_Sprint5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q "numpy<2.0"

!pip install -q --upgrade \
crewai \
litellm \
google-generativeai \
google-genai \
chromadb \
sentence-transformers \
crewai-tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 26.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xa

In [ ]:
import os, getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter Gemini API Key: ")

print("API key set")

Enter Gemini API Key: ··········
API key set


In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool as crewai_tool

from chromadb import PersistentClient
from sentence_transformers import SentenceTransformer

import json
import pathlib

In [ ]:
llm = "gemini/gemini-2.5-flash"

print(llm)

gemini/gemini-2.5-flash


In [ ]:
client_db = PersistentClient(path='./chroma_db')

col = client_db.get_or_create_collection('placement_kb')

embed = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

print("Vector DB ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector DB ready


In [ ]:
@crewai_tool
def rag_search(query: str) -> str:
    """
    Search placement knowledge base.
    """

    qv = embed.encode([query]).tolist()

    results = col.query(
        query_embeddings=qv,
        n_results=4
    )

    docs = results['documents'][0]

    return '\n---\n'.join(docs)

print("RAG tool created")

RAG tool created


In [ ]:
researcher = Agent(
    role="Placement Researcher",

    goal="Research company placement requirements for students",

    backstory="You prepare factual placement research notes using RAG search.",

    llm=llm,

    tools=[rag_search],

    verbose=True,

    allow_delegation=False,
)

print("Researcher agent ready")

Researcher agent ready


In [ ]:
interviewer = Agent(
    role="Mock Interviewer",

    goal="Generate placement interview questions",

    backstory="You generate technical and HR interview questions.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

print("Interviewer agent ready")

Interviewer agent ready


In [ ]:
coach = Agent(
    role="Answer Coach",

    goal="Create strong sample answers and preparation guidance",

    backstory="You coach students with strong placement answers.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

print("Coach agent ready")

Coach agent ready


In [ ]:
tracker = Agent(
    role="Progress Tracker",

    goal="Create structured student progress summaries",

    backstory="You generate JSON-style student progress summaries.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

print("Tracker agent ready")

Tracker agent ready


In [ ]:
profiles = [
    {
        "student_id": "S001",
        "name": "Ravi Kumar",
        "branch": "CSE",
        "cgpa": 7.8,
        "skills": ["Python", "Java", "SQL"],
        "target_company": "TCS Digital"
    },

    {
        "student_id": "S002",
        "name": "Sneha Reddy",
        "branch": "ECE",
        "cgpa": 8.1,
        "skills": ["Python", "DBMS"],
        "target_company": "Cognizant"
    },

    {
        "student_id": "S003",
        "name": "Arun Pillai",
        "branch": "IT",
        "cgpa": 8.5,
        "skills": ["Java", "DSA", "OOPs"],
        "target_company": "Amazon"
    }
]

print("Profiles loaded")

Profiles loaded


In [ ]:
def build_tasks(profile):

    research = Task(
        description=(
            f"Research placement preparation details for "
            f"{profile['target_company']} "
            f"for a {profile['branch']} student."
        ),

        agent=researcher,

        expected_output="Short bullet research notes."
    )

    interview = Task(
        description=(
            f"Generate EXACTLY 10 mock interview questions "
            f"for {profile['name']} targeting "
            f"{profile['target_company']}."
        ),

        agent=interviewer,

        expected_output="Exactly 10 numbered interview questions."
    )

    coaching = Task(
        description=(
            f"Pick question 3 and create strong sample answer "
            f"for {profile['name']}."
        ),

        agent=coach,

        expected_output="One strong answer and 3 preparation tips."
    )

    tracking = Task(
        description=(
            f"Create JSON-style progress summary "
            f"for {profile['student_id']}."
        ),

        agent=tracker,

        expected_output="Valid JSON-style summary."
    )

    return [research, interview, coaching, tracking]

print("Task builder ready")

Task builder ready


In [ ]:
p = profiles[0]

crew = Crew(
    agents=[
        researcher,
        interviewer,
        coach,
        tracker
    ],

    tasks=build_tasks(p),

    process=Process.sequential,

    verbose=True,
)

print("Crew created")

Crew created


In [ ]:
result = await crew.kickoff_async()

print("\n=== FINAL OUTPUT ===\n")

print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3fc71359-1a10-4e9c-9f08-c76091640a43                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research placement preparation details for TCS Digital for a CSE student.                                │
│  ID: 43626e47-7d5d-4fef-b3b0-38ee14489bc8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Placement Researcher                                                                                    │
│                                                                                                                 │
│  Task: Research placement preparation details for TCS Digital for a CSE student.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#4) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: rag_search                                                                                               │
│  Args: {'query': 'TCS Digital placement preparation for CSE students'}                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool rag_search executed with result: ...


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Progress Tracker                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```json                                                                                                        │
│  {                                                                                                              │
│    "student_id": "S001",                                                                                        │
│    "student_name": "Ravi Kumar",                                                                                │
│    "interview_focus": "TCS Digital",                                                                            │
│    "progress_summary": "Ravi Kumar is preparing for the TCS Digital interview. A strong sample answer was       │
│  provided for a key question on digital transformation technologies, demonstrating a solid understanding of     │
│  relevant concepts and practical experience. Further refinement based on company-specific offerings and         │
│  continued practice across all mock questions is recommended.",                                                 │
│    "areas_of_strength_demonstrated": [                                                                          │
│      "Comprehensive understanding of key digital transformation technologies (Cloud Computing, AI/ML, Data      │
│  Analytics, DevOps/Containerization).",                                                                         │
│      "Ability to link technologies to business value and impact (scalability, agility, decision-making,         │
│  efficiency).",                                                                                                 │
│      "Strong articulation of practical experience with specific tools and platforms (AWS, Python,               │
│  Scikit-learn, TensorFlow, SQL, Pandas, Matplotlib, Seaborn, Docker).",                                         │
│      "Clear communication and structured response for complex technical questions."                             │
│    ],                                                                                                           │
│    "areas_for_further_development": [                                                                           │
│      "Tailoring answers specifically to TCS Digital's unique service lines, case studies, and terminology.",    │
│      "Ensuring consistency in connecting all technological discussions to broader business value.",             │
│      "Expanding and deepening practical examples for other technical domains relevant to TCS Digital."          │
│    ],                                                                                                           │
│    "recommendations": [                                                                                         │
│      "Deeply research TCS Digital's specific offerings, projects, and strategic priorities to align answers     │
│  with their organizational language and focus.",                                                                │
│      "Practice articulating the business impact and value proposition of technologies for various scenarios,    │
│  not just what they are.",                                                                                      │
│      "Continue to provide concrete, project-based examples for all technical skills and experiences mentioned,  │
│  detailing 'what was done' and 'what was learned'.",   

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Create JSON-style progress summary for S001.                                                             │
│  Agent: Progress Tracker                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== FINAL OUTPUT ===

```json
{
  "student_id": "S001",
  "student_name": "Ravi Kumar",
  "interview_focus": "TCS Digital",
  "progress_summary": "Ravi Kumar is preparing for the TCS Digital interview. A strong sample answer was provided for a key question on digital transformation technologies, demonstrating a solid understanding of relevant concepts and practical experience. Further refinement based on company-specific offerings and continued practice across all mock questions is recommended.",
  "areas_of_strength_demonstrated": [
    "Comprehensive understanding of key digital transformation technologies (Cloud Computing, AI/ML, Data Analytics, DevOps/Containerization).",
    "Ability to link technologies to business value and impact (scalability, agility, decision-making, efficiency).",
    "Strong articulation of practical experience with specific tools and platforms (AWS, Python, Scikit-learn, TensorFlow, SQL, Pandas, Matplotlib, Seaborn, Docker).",
    "Clear communication and

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 3fc71359-1a10-4e9c-9f08-c76091640a43                                                                       │
│  Final Output: ```json                                                                                          │
│  {                                                                                                              │
│    "student_id": "S001",                                                                                        │
│    "student_name": "Ravi Kumar",                                                                                │
│    "interview_focus": "TCS Digital",                                                                            │
│    "progress_summary": "Ravi Kumar is preparing for the TCS Digital interview. A strong sample answer was       │
│  provided for a key question on digital transformation technologies, demonstrating a solid understanding of     │
│  relevant concepts and practical experience. Further refinement based on company-specific offerings and         │
│  continued practice across all mock questions is recommended.",                                                 │
│    "areas_of_strength_demonstrated": [                                                                          │
│      "Comprehensive understanding of key digital transformation technologies (Cloud Computing, AI/ML, Data      │
│  Analytics, DevOps/Containerization).",                                                                         │
│      "Ability to link technologies to business value and impact (scalability, agility, decision-making,         │
│  efficiency).",                                                                                                 │
│      "Strong articulation of practical experience with specific tools and platforms (AWS, Python,               │
│  Scikit-learn, TensorFlow, SQL, Pandas, Matplotlib, Seaborn, Docker).",                                         │
│      "Clear communication and structured response for complex technical questions."                             │
│    ],                                                                                                           │
│    "areas_for_further_development": [                                                                           │
│      "Tailoring answers specifically to TCS Digital's unique service lines, case studies, and terminology.",    │
│      "Ensuring consistency in connecting all technological discussions to broader business value.",             │
│      "Expanding and deepening practical examples for other technical domains relevant to TCS Digital."          │
│    ],                                                                                                           │
│    "recommendations": [                                                                                         │
│      "Deeply research TCS Digital's specific offerings, projects, and strategic priorities to align answers     │
│  with their organizational language and focus.",                                                                │
│      "Practice articulating the business impact and value proposition of technologies for various scenarios,    │
│  not just what they are.",                                                                                      │
│      "Continue to provide concrete, project-based examples for all technical skills and experiences mentioned,  │
│  detailing 'what was done' and 'what was learned'.",  

In [27]:
import asyncio
import time

transcripts = []

async def run_safe_crew(p):

    try:
        print("\n" + "="*60)
        print(f"Running for {p['name']} → {p['target_company']}")
        print("="*60)

        crew = Crew(
            agents=[researcher, interviewer, coach, tracker],
            tasks=build_tasks(p),
            process=Process.sequential,
            verbose=True,
        )

        result = await crew.kickoff_async()

        return {
            "student": p["name"],
            "target": p["target_company"],
            "final_output": str(result),
            "status": "success"
        }

    except Exception as e:

        print("\n❌ Crew failed for:", p["name"])
        print("Error:", str(e)[:300])

        # ⛔ wait for quota reset (important for Gemini 429)
        print("⏳ Waiting 30 seconds before next run...")
        time.sleep(30)

        return {
            "student": p["name"],
            "target": p["target_company"],
            "final_output": f"FAILED: {str(e)[:200]}",
            "status": "failed"
        }


# =========================
# MAIN LOOP (SAFE EXECUTION)
# =========================

for p in profiles:

    result = await run_safe_crew(p)
    transcripts.append(result)

print("\nCompleted:", len(transcripts))


Running for Ravi Kumar → TCS Digital


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: faee333e-a166-4fc5-b3b7-6b1894e2da60                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research placement preparation details for TCS Digital for a CSE student.                                │
│  ID: 3ebd1554-a080-493b-80f2-9ad4cf0364f0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Placement Researcher                                                                                    │
│                                                                                                                 │
│  Task: Research placement preparation details for TCS Digital for a CSE student.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:root:Google Gemini API error: 404 - models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.
ERROR:crewai.flow.flow:Error executing listener call_llm_native_tools: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}


...


❌ Crew failed for: Arun Pillai
Error: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
⏳ Waiting 30 seconds before next run...


╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Research placement preparation details for Amazon for a IT student.                                      │
│  Agent: Placement Researcher                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 41d51335-1a87-44f4-b7de-5b4a09b5c002                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Completed: 3


In [28]:
pathlib.Path(
    "day10_sprint5_transcripts.json"
).write_text(
    json.dumps(transcripts, indent=2)
)

print("Saved transcripts successfully")

Saved transcripts successfully


In [29]:
md = "# Day10 Sprint5 Report\n\n"

for t in transcripts:

    md += f"## {t['student']} → {t['target']}\n\n"

    md += t["final_output"]

    md += "\n\n---\n\n"

pathlib.Path(
    "day10_sprint5_report.md"
).write_text(md)

print("Markdown report created")

Markdown report created


In [30]:
from google.colab import files

files.download("day10_sprint5_transcripts.json")

files.download("day10_sprint5_report.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>